# Real Data Preparation

This notebook prepares a first real/public dataset for the MAL part of the project.

The goal is not to train the final model yet. The goal is to clean the two Kaggle datasets, keep the columns that match our project, rename them into our project format, and create a prepared dataset that looks similar to our current `focus_dataset.csv`.

Important limitation: the two Kaggle datasets do not describe the exact same rooms or sessions, so this is not a true database join. We combine them as an exploratory dataset so we can move away from fully mock-generated data and test model ideas with more realistic room and focus values.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

notebook_dir = Path.cwd()
room_path = notebook_dir / "room_occupancy_kaggle.csv"
noise_path = notebook_dir / "background_noise_focus_kaggle.csv"

room_raw = pd.read_csv(room_path)
noise_raw = pd.read_csv(noise_path)

print("Room occupancy shape:", room_raw.shape)
print("Noise/focus shape:", noise_raw.shape)

display(room_raw.head())
display(noise_raw.head())


Room occupancy shape: (2665, 6)
Noise/focus shape: (500, 10)


,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,23.7000,26.272,585.200000,749.200000,0.004764,1
1,23.7180,26.290,578.400000,760.400000,0.004773,1
2,23.7300,26.230,572.666667,769.666667,0.004765,1
3,23.7225,26.125,493.750000,774.750000,0.004744,1
4,23.7540,26.200,488.600000,779.000000,0.004767,1


,participant_id,age,role,task_type,background_noise_type,noise_volume_level,focus_duration_minutes,perceived_focus_score,task_completion_quality,mental_fatigue_after_task
0,1,44,Student,Writing,Traffic Noise,8,98,5,5,1
1,2,30,Student,Writing,Silence,9,31,3,9,7
2,3,23,Remote Worker,Reading,Silence,4,39,9,1,10
3,4,36,Professional,Writing,Songs with Lyrics,1,93,4,4,10
4,5,34,Remote Worker,Studying,Traffic Noise,1,94,6,3,2


In [2]:
print("Room occupancy columns:")
display(room_raw.dtypes)

print("Missing values in room occupancy data:")
display(room_raw.isna().sum())

print("Noise/focus columns:")
display(noise_raw.dtypes)

print("Missing values in noise/focus data:")
display(noise_raw.isna().sum())


Room occupancy columns:


Temperature      float64
Humidity         float64
Light            float64
CO2              float64
HumidityRatio    float64
Occupancy          int64
dtype: object

Missing values in room occupancy data:


Temperature      0
Humidity         0
Light            0
CO2              0
HumidityRatio    0
Occupancy        0
dtype: int64

Noise/focus columns:


participant_id                int64
age                           int64
role                         object
task_type                    object
background_noise_type        object
noise_volume_level            int64
focus_duration_minutes        int64
perceived_focus_score         int64
task_completion_quality       int64
mental_fatigue_after_task     int64
dtype: object

Missing values in noise/focus data:


participant_id               0
age                          0
role                         0
task_type                    0
background_noise_type        0
noise_volume_level           0
focus_duration_minutes       0
perceived_focus_score        0
task_completion_quality      0
mental_fatigue_after_task    0
dtype: int64

In [3]:
# Keep only the columns that match our project sensors.
# These names are changed to be close to our database/backend naming.

room_clean = room_raw.rename(columns={
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Light": "light_level",
    "CO2": "co2_level",
    "HumidityRatio": "humidity_ratio",
    "Occupancy": "occupancy",
})

room_clean = room_clean[
    ["temperature", "humidity", "light_level", "co2_level", "humidity_ratio", "occupancy"]
].copy()

numeric_columns = ["temperature", "humidity", "light_level", "co2_level", "humidity_ratio", "occupancy"]

for column in numeric_columns:
    room_clean[column] = pd.to_numeric(room_clean[column], errors="coerce")

room_clean = room_clean.dropna().drop_duplicates().reset_index(drop=True)

display(room_clean.head())
display(room_clean.describe())


,temperature,humidity,light_level,co2_level,humidity_ratio,occupancy
0,23.7000,26.272,585.200000,749.200000,0.004764,1
1,23.7180,26.290,578.400000,760.400000,0.004773,1
2,23.7300,26.230,572.666667,769.666667,0.004765,1
3,23.7225,26.125,493.750000,774.750000,0.004744,1
4,23.7540,26.200,488.600000,779.000000,0.004767,1


,temperature,humidity,light_level,co2_level,humidity_ratio,occupancy
count,2583.000000,2583.000000,2583.000000,2583.000000,2583.000000,2583.000000
mean,21.460902,25.416843,199.361764,725.847461,0.004044,0.376307
std,1.032147,2.439779,251.734668,293.717077,0.000612,0.484552
min,20.200000,22.100000,0.000000,427.500000,0.003303,0.000000
25%,20.675000,23.390000,0.000000,467.400000,0.003558,0.000000
50%,20.972500,25.000000,0.000000,594.250000,0.003830,0.000000
75%,22.390000,27.000000,444.000000,970.791667,0.004545,1.000000
max,24.408333,31.472500,1697.250000,1402.250000,0.005378,1.000000


In [4]:
# Keep the focus-related columns and rename them into simpler project-style names.

noise_clean = noise_raw.rename(columns={
    "background_noise_type": "noise_type",
    "noise_volume_level": "noise_volume_level",
    "focus_duration_minutes": "focus_duration_minutes",
    "perceived_focus_score": "perceived_focus_score",
    "task_completion_quality": "task_completion_quality",
    "mental_fatigue_after_task": "mental_fatigue_after_task",
})

noise_clean = noise_clean[
    [
        "role",
        "task_type",
        "noise_type",
        "noise_volume_level",
        "focus_duration_minutes",
        "perceived_focus_score",
        "task_completion_quality",
        "mental_fatigue_after_task",
    ]
].copy()

for column in [
    "noise_volume_level",
    "focus_duration_minutes",
    "perceived_focus_score",
    "task_completion_quality",
    "mental_fatigue_after_task",
]:
    noise_clean[column] = pd.to_numeric(noise_clean[column], errors="coerce")

noise_clean = noise_clean.dropna().drop_duplicates().reset_index(drop=True)

# Our project currently uses noiseLevel in mock data.
# The Kaggle noise volume is 1-10, so this is only an approximate conversion to a dB-like value.
noise_clean["noise_level"] = 30 + noise_clean["noise_volume_level"] * 5

display(noise_clean.head())
display(noise_clean.describe())


,role,task_type,noise_type,noise_volume_level,focus_duration_minutes,perceived_focus_score,task_completion_quality,mental_fatigue_after_task,noise_level
0,Student,Writing,Traffic Noise,8,98,5,5,1,70
1,Student,Writing,Silence,9,31,3,9,7,75
2,Remote Worker,Reading,Silence,4,39,9,1,10,50
3,Professional,Writing,Songs with Lyrics,1,93,4,4,10,35
4,Remote Worker,Studying,Traffic Noise,1,94,6,3,2,35


,noise_volume_level,focus_duration_minutes,perceived_focus_score,task_completion_quality,mental_fatigue_after_task,noise_level
count,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000
mean,5.440000,63.898000,5.254000,5.59400,5.374000,57.200000
std,2.848619,33.004214,2.820886,2.82154,2.992664,14.243094
min,1.000000,10.000000,1.000000,1.00000,1.000000,35.000000
25%,3.000000,33.750000,3.000000,3.00000,3.000000,45.000000
50%,5.000000,63.500000,5.000000,6.00000,5.000000,55.000000
75%,8.000000,94.000000,8.000000,8.00000,8.000000,70.000000
max,10.000000,120.000000,10.000000,10.00000,10.000000,80.000000


In [5]:
# The datasets have no shared session id or timestamp.
# To "glue" them, we sample/repeat focus rows so every room sensor row gets one focus/noise row.
# This is exploratory, not perfect ground truth.

noise_aligned = noise_clean.sample(
    n=len(room_clean),
    replace=len(noise_clean) < len(room_clean),
    random_state=RANDOM_STATE,
).reset_index(drop=True)

combined = pd.concat([room_clean.reset_index(drop=True), noise_aligned], axis=1)

# Create synthetic session/time information so the dataset can be shaped like our project data.
SESSION_SIZE = 30

combined["session_id"] = (combined.index // SESSION_SIZE) + 1
combined["sent_at"] = pd.date_range("2026-04-01 09:00:00", periods=len(combined), freq="min")

display(combined.head())
print("Combined rows:", len(combined))
print("Sessions:", combined["session_id"].nunique())


,temperature,humidity,light_level,co2_level,humidity_ratio,occupancy,role,task_type,noise_type,noise_volume_level,focus_duration_minutes,perceived_focus_score,task_completion_quality,mental_fatigue_after_task,noise_level,session_id,sent_at
0,23.7000,26.272,585.200000,749.200000,0.004764,1,Remote Worker,Writing,Songs with Lyrics,10,99,8,3,1,80,1,2026-04-01 09:00:00
1,23.7180,26.290,578.400000,760.400000,0.004773,1,Student,Reading,Traffic Noise,1,64,4,9,9,35,1,2026-04-01 09:01:00
2,23.7300,26.230,572.666667,769.666667,0.004765,1,Remote Worker,Writing,Songs with Lyrics,4,48,10,4,9,50,1,2026-04-01 09:02:00
3,23.7225,26.125,493.750000,774.750000,0.004744,1,Professional,Reading,Songs with Lyrics,2,44,3,4,10,40,1,2026-04-01 09:03:00
4,23.7540,26.200,488.600000,779.000000,0.004767,1,Professional,Studying,Songs with Lyrics,4,90,8,8,1,50,1,2026-04-01 09:04:00


Combined rows: 2583
Sessions: 87


In [6]:
# Create a target that can support both approaches:
# - rating_score: continuous value for regression
# - rating: class value from 1 to 5 for classification and frontend decisions

def minmax_scale_1_to_10(series):
    series = pd.to_numeric(series, errors="coerce")
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(5, index=series.index)

    return 1 + 9 * ((series - minimum) / (maximum - minimum))


duration_score = minmax_scale_1_to_10(combined["focus_duration_minutes"])

combined["focus_score_1_to_10"] = (
    0.45 * combined["perceived_focus_score"]
    + 0.25 * combined["task_completion_quality"]
    + 0.20 * (11 - combined["mental_fatigue_after_task"])
    + 0.10 * duration_score
)

base_rating_score = 1 + 4 * ((combined["focus_score_1_to_10"] - 1) / 9)

environment_penalty = (
    0.40 * ((combined["temperature"] < 19) | (combined["temperature"] > 25)).astype(int)
    + 0.30 * ((combined["humidity"] < 30) | (combined["humidity"] > 60)).astype(int)
    + 0.50 * (combined["co2_level"] > 1000).astype(int)
    + 0.50 * (combined["co2_level"] > 1400).astype(int)
    + 0.30 * (combined["light_level"] < 100).astype(int)
    + 0.20 * (combined["noise_volume_level"] > 6).astype(int)
)

combined["rating_score"] = (base_rating_score - environment_penalty).clip(1, 5).round(2)
combined["rating"] = combined["rating_score"].round().clip(1, 5).astype(int)

# Simple frontend interpretation for later discussion.
combined["stay_or_leave"] = np.where(combined["rating"] >= 3, "stay", "leave")

display(combined[[
    "temperature",
    "humidity",
    "co2_level",
    "light_level",
    "noise_level",
    "rating_score",
    "rating",
    "stay_or_leave",
]].head())

display(combined["rating"].value_counts().sort_index().rename("rows_per_rating"))


,temperature,humidity,co2_level,light_level,noise_level,rating_score,rating,stay_or_leave
0,23.7000,26.272,749.200000,585.200000,80,3.25,3,stay
1,23.7180,26.290,760.400000,578.400000,35,2.47,2,leave
2,23.7300,26.230,769.666667,572.666667,50,3.06,3,stay
3,23.7225,26.125,774.750000,493.750000,40,1.56,2,leave
4,23.7540,26.200,779.000000,488.600000,50,3.97,4,stay


rating
1     347
2    1144
3     952
4     140
Name: rows_per_rating, dtype: int64

In [7]:
# This version looks similar to the raw database table used by the IoT backend.
# The real database currently has temperature, humidity, co2_level, light_level, session_id, and sent_at.
# noise_level and occupancy are useful extra columns for ML exploration.

environment_history_public = combined[
    [
        "session_id",
        "sent_at",
        "temperature",
        "humidity",
        "co2_level",
        "light_level",
        "noise_level",
        "occupancy",
        "rating_score",
        "rating",
    ]
].copy()

display(environment_history_public.head())


,session_id,sent_at,temperature,humidity,co2_level,light_level,noise_level,occupancy,rating_score,rating
0,1,2026-04-01 09:00:00,23.7000,26.272,749.200000,585.200000,80,1,3.25,3
1,1,2026-04-01 09:01:00,23.7180,26.290,760.400000,578.400000,35,1,2.47,2
2,1,2026-04-01 09:02:00,23.7300,26.230,769.666667,572.666667,50,1,3.06,3
3,1,2026-04-01 09:03:00,23.7225,26.125,774.750000,493.750000,40,1,1.56,2
4,1,2026-04-01 09:04:00,23.7540,26.200,779.000000,488.600000,50,1,3.97,4


In [8]:
# Aggregate rows into session-level features.
# This creates a dataset shaped like our current MAL/app/focus_dataset.csv.

rows = []

for session_id, session in combined.groupby("session_id"):
    start_time = session["sent_at"].min()
    end_time = session["sent_at"].max()
    mean_rating_score = session["rating_score"].mean()

    rows.append({
        "timePeriod": f"{start_time} to {end_time}",

        "currentTemperature": session["temperature"].iloc[-1],
        "currentHumidity": session["humidity"].iloc[-1],
        "currentCO2": session["co2_level"].iloc[-1],
        "currentLight": session["light_level"].iloc[-1],
        "currentNoise": session["noise_level"].iloc[-1],

        "maxTemp": session["temperature"].max(),
        "minTemp": session["temperature"].min(),
        "meanTemp": round(session["temperature"].mean(), 2),

        "maxHumidity": session["humidity"].max(),
        "minHumidity": session["humidity"].min(),
        "meanHumidity": round(session["humidity"].mean(), 2),

        "maxCO2": session["co2_level"].max(),
        "minCO2": session["co2_level"].min(),
        "meanCO2": round(session["co2_level"].mean(), 2),

        "maxLight": session["light_level"].max(),
        "minLight": session["light_level"].min(),
        "meanLight": round(session["light_level"].mean(), 2),

        "maxNoise": session["noise_level"].max(),
        "minNoise": session["noise_level"].min(),
        "meanNoise": round(session["noise_level"].mean(), 2),

        "rating_score": round(mean_rating_score, 2),
        "rating": int(np.clip(round(mean_rating_score), 1, 5)),
    })

focus_dataset_public = pd.DataFrame(rows)

display(focus_dataset_public.head())
print("Prepared focus dataset rows:", len(focus_dataset_public))
display(focus_dataset_public["rating"].value_counts().sort_index().rename("rows_per_rating"))


,timePeriod,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,maxTemp,minTemp,meanTemp,maxHumidity,...,minCO2,meanCO2,maxLight,minLight,meanLight,maxNoise,minNoise,meanNoise,rating_score,rating
0,2026-04-01 09:00:00 to 2026-04-01 09:29:00,23.600000,27.50,965.333333,522.0,45,23.760000,23.600,23.68,27.500,...,749.200000,861.50,585.200000,454.0,501.56,80,35,55.67,2.88,3
1,2026-04-01 09:30:00 to 2026-04-01 09:59:00,23.370000,28.39,1090.600000,464.0,50,23.666667,23.370,23.52,28.390,...,978.428571,1037.16,520.500000,439.0,470.99,80,35,57.67,1.99,2
2,2026-04-01 10:00:00 to 2026-04-01 10:29:00,23.150000,28.89,1159.500000,448.0,40,23.390000,23.140,23.26,28.890,...,1086.000000,1123.15,475.666667,446.0,462.59,80,35,56.67,2.24,2
3,2026-04-01 10:30:00 to 2026-04-01 10:59:00,22.873333,27.60,1056.400000,429.0,80,23.180000,22.865,23.00,29.075,...,1042.600000,1094.90,444.000000,429.0,433.86,80,35,57.67,2.24,2
4,2026-04-01 11:00:00 to 2026-04-01 11:29:00,22.666667,25.60,892.000000,429.0,50,22.890000,22.600,22.73,27.600,...,892.000000,961.64,444.000000,429.0,436.89,80,35,59.67,2.21,2


Prepared focus dataset rows: 87


rating
2    70
3    17
Name: rows_per_rating, dtype: int64

In [9]:
# Final validation checks before saving or using the data.

required_current_model_columns = ["currentTemperature", "maxTemp", "minTemp", "meanTemp", "rating"]
required_future_columns = [
    "currentHumidity",
    "currentCO2",
    "currentLight",
    "currentNoise",
    "meanHumidity",
    "meanCO2",
    "meanLight",
    "meanNoise",
]

missing_current = [column for column in required_current_model_columns if column not in focus_dataset_public.columns]
missing_future = [column for column in required_future_columns if column not in focus_dataset_public.columns]

print("Missing columns for current model:", missing_current)
print("Missing columns for future expanded model:", missing_future)
print("Any missing values:", focus_dataset_public.isna().sum().sum())

display(focus_dataset_public.describe())


Missing columns for current model: []
Missing columns for future expanded model: []
Any missing values: 0


,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,maxTemp,minTemp,meanTemp,maxHumidity,minHumidity,...,minCO2,meanCO2,maxLight,minLight,meanLight,maxNoise,minNoise,meanNoise,rating_score,rating
count,87.000000,87.000000,87.000000,87.000000,87.00000,87.000000,87.000000,87.000000,87.000000,87.000000,...,87.000000,87.000000,87.000000,87.000000,87.000000,87.000000,87.000000,87.000000,87.000000,87.000000
mean,21.496830,25.405084,731.835441,224.453448,54.54023,21.585165,21.401422,21.490805,25.656134,25.182103,...,702.370252,729.968966,252.296552,192.036371,205.671839,79.885057,35.344828,57.252989,2.339655,2.195402
std,1.083950,2.443087,297.805669,287.945704,13.92739,1.110469,1.038622,1.071463,2.539251,2.343384,...,281.746226,295.728299,329.277036,245.311919,255.101920,0.753678,1.835234,2.436886,0.175423,0.398809
min,20.200000,22.100000,430.000000,0.000000,35.00000,20.323333,20.200000,20.260000,22.200000,22.100000,...,427.500000,433.690000,0.000000,0.000000,0.000000,75.000000,35.000000,53.000000,1.970000,2.000000
25%,20.650000,23.465000,467.875000,0.000000,45.00000,20.700000,20.622500,20.675000,23.800000,23.265000,...,460.291667,467.160000,0.000000,0.000000,0.000000,80.000000,35.000000,55.500000,2.210000,2.000000
50%,21.000000,24.978000,605.666667,0.000000,50.00000,21.080000,20.890000,20.970000,25.175000,24.890000,...,582.250000,599.440000,0.000000,0.000000,0.000000,80.000000,35.000000,56.830000,2.320000,2.000000
75%,22.512500,26.840000,979.541667,446.000000,65.00000,22.600000,22.245000,22.410000,27.500000,26.298750,...,920.800000,988.120000,460.000000,435.500000,441.665000,80.000000,35.000000,59.085000,2.425000,2.000000
max,24.408333,31.315000,1388.000000,1419.500000,80.00000,24.408333,24.330000,24.360000,31.472500,31.166667,...,1365.333333,1388.850000,1697.250000,798.000000,809.330000,80.000000,50.000000,63.330000,2.880000,3.000000


In [10]:
# Optional save step.
# Running this creates prepared CSV files inside MAL/Jupyter_Notebooks/prepared_data.

output_dir = notebook_dir / "prepared_data"
output_dir.mkdir(exist_ok=True)

environment_output_path = output_dir / "environment_history_public_prepared.csv"
focus_output_path = output_dir / "focus_dataset_public_prepared.csv"

environment_history_public.to_csv(environment_output_path, index=False)
focus_dataset_public.to_csv(focus_output_path, index=False)

print("Saved:", environment_output_path)
print("Saved:", focus_output_path)


Saved: c:\Users\mahru\OneDrive\SCHOOL\SEP4\SEP4\MAL\Jupyter_Notebooks\prepared_data\environment_history_public_prepared.csv
Saved: c:\Users\mahru\OneDrive\SCHOOL\SEP4\SEP4\MAL\Jupyter_Notebooks\prepared_data\focus_dataset_public_prepared.csv


## Notes

This prepared dataset is a first step toward replacing fully generated mock data with public real-world data. The room occupancy dataset gives realistic environmental sensor values: temperature, humidity, light, and CO2. The background noise/focus dataset gives focus-related values such as perceived focus score, task quality, fatigue, and noise level.

Because the datasets do not share the same participants, rooms, or timestamps, they cannot be joined as true real sessions. Instead, this notebook aligns them into an exploratory dataset that follows the same structure as our project data. This gives us a better starting point for testing models while we wait for enough real data from our own IoT/backend system.

The prepared data keeps two target variables:
- `rating_score`, which can be used for regression.
- `rating`, which turns the result into classes from 1 to 5.

For the current frontend, the class rating is probably the most useful output because it can be mapped directly to a simple stay/leave recommendation. Regression is still worth testing because the rounded prediction may give smoother results than forcing the model to learn only fixed classes.